# Pré-processamento e Feature Engineering

Pipeline que gera o dataset final usado no modelo supervisionado.

**Formato de saída** — um arquivo Parquet em `data/model_dataset.parquet`:

| coluna | tipo | descrição |
|---|---|---|
| `stop_id` | str | identificador da parada (GTFS) |
| `ano_mes` | str `YYYY-MM` | mês de referência |
| `lat`, `lon` | float | geografia da parada |
| `num_rotas_servindo` | int | quantas rotas atendem essa parada |
| `pagerank` | float | importância na rede de transporte (calculada in-notebook) |
| `num_reclamacoes_3m/6m/12m` | int | total de reclamações 1746 vinculadas nos últimos K meses |
| `num_reclamacoes_seguranca_3m/6m/12m` | int | idem, filtrado por tipo relacionado a Segurança Pública |
| `num_reclamacoes_iluminacao_3m/6m/12m` | int | idem, tipo relacionado a Iluminação Pública |
| `mes` | int (1–12) | mês do ano — codificação cíclica (sin/cos) fica no notebook do modelo |
| `ano` | int | tendência |
| `y` | int (0/1) | **target**: 1 se houve ≥1 tiroteio dentro de 500m da parada no mês |

**Decisões metodológicas** (ver conversa acima):
- Raio do target: **500m** (Fogo Cruzado — target denso o suficiente para aprender).
- Vinculação de reclamação → parada: **`stop_id_mais_proximo`** já pré-computado no CSV (raio ~100m do paper).
- Sem features derivadas do Fogo Cruzado (só ele é o target — a pergunta científica é se reclamações + rede predizem violência).
- Sem `bairro` como feature (geografia via lat/lon).
- Janelas de reclamação: 3, 6, 12 meses. Sem 1m (ruído).
- **Zero leakage**: todas as features de um mês M usam apenas dados com `data < início do mês M`.

**Corte temporal**: Fogo Cruzado começa em 2019-12, 1746 começa em 2020-01. Com warmup de 12m, o dataset final vai de **2021-01 a 2025-12** (60 meses × ~6.4k paradas ≈ 385k linhas).

In [1]:
import numpy as np
import pandas as pd
import networkx as nx
from pathlib import Path
from sklearn.neighbors import BallTree

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 220)

REPO = Path.cwd()
CHAMADOS_CSV = REPO / 'data' / '1746' / 'chamados_v2_com_stops_filtrado.csv'
FOGO_CSV = REPO / 'data' / 'fogocruzado' / 'fc_api_occurrences_with_victims_2026-08-01T12_38_31.000Z.csv'
STOPS_TXT = REPO / 'data' / 'gtfs' / 'stops.txt'
STOP_TIMES_TXT = REPO / 'data' / 'gtfs' / 'stop_times.txt'
TRIPS_TXT = REPO / 'data' / 'gtfs' / 'trips.txt'

OUTPUT = REPO / 'data' / 'model_dataset.parquet'

RAIO_TARGET_M = 500      # raio do target (tiroteio -> parada)
WARMUP_MESES = 12        # meses de histórico necessários antes do primeiro mês do dataset
R_TERRA_M = 6_371_000.0  # raio da Terra em metros (BallTree haversine)

## 1. Carga das bases

In [2]:
chamados = pd.read_csv(CHAMADOS_CSV, parse_dates=['data_inicio'])
chamados = chamados.dropna(subset=['stop_id_mais_proximo', 'data_inicio']).copy()
chamados['ano_mes'] = chamados['data_inicio'].dt.to_period('M')
chamados['stop_id_mais_proximo'] = chamados['stop_id_mais_proximo'].astype(str)

print(f'Chamados: {len(chamados):,}')
print(f'Período: {chamados["data_inicio"].min()} - {chamados["data_inicio"].max()}')
print()
print('Top 10 tipos:')
print(chamados["tipo"].value_counts().head(10))

Chamados: 47,680
Período: 2020-01-02 08:37:45 - 2025-12-13 00:26:00

Top 10 tipos:
tipo
Alvará                                         22218
Diversos - Comlurb                              9028
Iluminação Pública                              6104
Guarda Municipal / Fiscalização de trânsito     2785
Estacionamento                                  2492
Conservação de vias                             1885
Vias públicas                                   1307
Ônibus                                           772
Ordem pública                                    294
Regulamentações Viárias                          277
Name: count, dtype: int64


In [3]:
fogo = pd.read_csv(FOGO_CSV)
fogo['data'] = pd.to_datetime(fogo['data'], format='%d/%m/%Y, %H:%M:%S', errors='coerce')
fogo_rio = fogo[
    (fogo['city'] == 'Rio de Janeiro')
    & fogo['data'].notna()
    & fogo['latitude'].notna()
    & fogo['longitude'].notna()
].copy()
fogo_rio['ano_mes'] = fogo_rio['data'].dt.to_period('M')

print(f'Tiroteios (Rio, coords + data válidas): {len(fogo_rio):,}')
print(f'Período: {fogo_rio["data"].min()} - {fogo_rio["data"].max()}')

Tiroteios (Rio, coords + data válidas): 12,637
Período: 2019-12-31 21:01:00 - 2025-12-31 14:05:00


In [4]:
stops = pd.read_csv(STOPS_TXT, usecols=['stop_id', 'stop_lat', 'stop_lon'])
stops = stops.dropna(subset=['stop_lat', 'stop_lon']).drop_duplicates('stop_id').reset_index(drop=True)
stops['stop_id'] = stops['stop_id'].astype(str)
print(f'Paradas: {len(stops):,}')

Paradas: 6,426


## 2. Features estáticas da parada

Não variam com o tempo. Calculadas uma vez para cada parada.

In [5]:
# num_rotas_servindo = quantas rotas distintas passam pela parada
# stop_times.txt tem (trip_id, stop_id). Cada trip pertence a uma route via trips.txt.

stop_times = pd.read_csv(STOP_TIMES_TXT, usecols=['trip_id', 'stop_id'], dtype={'stop_id': str, 'trip_id': str})
trips = pd.read_csv(TRIPS_TXT, usecols=['trip_id', 'route_id'], dtype={'trip_id': str, 'route_id': str})

stop_route = stop_times.merge(trips, on='trip_id', how='left').dropna(subset=['route_id'])
num_rotas = stop_route.groupby('stop_id')['route_id'].nunique().rename('num_rotas_servindo')

stops = stops.merge(num_rotas, on='stop_id', how='left')
stops['num_rotas_servindo'] = stops['num_rotas_servindo'].fillna(0).astype(int)

print(f'num_rotas_servindo - min: {stops["num_rotas_servindo"].min()}, '
      f'max: {stops["num_rotas_servindo"].max()}, '
      f'média: {stops["num_rotas_servindo"].mean():.2f}')

num_rotas_servindo - min: 0, max: 60, média: 3.90


In [6]:
# pagerank sobre a rede de trânsito.
# Aresta (u, v) se u e v aparecem em sequência consecutiva em alguma trip.
# Mesma definição de CONNECTS_TO usada no Neo4j (scripts/02_load_gtfs_to_neo4j.py).

stop_times_seq = pd.read_csv(
    STOP_TIMES_TXT,
    usecols=['trip_id', 'stop_id', 'stop_sequence'],
    dtype={'trip_id': str, 'stop_id': str},
).sort_values(['trip_id', 'stop_sequence'])

# Para cada trip, gerar pares consecutivos (stop[i], stop[i+1])
grp = stop_times_seq.groupby('trip_id', sort=False)['stop_id']
edges = set()
for _, seq in grp:
    lst = seq.tolist()
    for u, v in zip(lst[:-1], lst[1:]):
        if u != v:
            edges.add((u, v))

print(f'Arestas únicas: {len(edges):,}')

G = nx.DiGraph()
G.add_nodes_from(stops['stop_id'].tolist())
G.add_edges_from(edges)
print(f'Nós: {G.number_of_nodes():,}, arestas: {G.number_of_edges():,}')

pr = nx.pagerank(G, alpha=0.85, max_iter=100)
pagerank_series = pd.Series(pr, name='pagerank')
pagerank_series.index.name = 'stop_id'
stops = stops.merge(pagerank_series.reset_index(), on='stop_id', how='left')
stops['pagerank'] = stops['pagerank'].fillna(0.0)

print(f'\npagerank - min: {stops["pagerank"].min():.2e}, '
      f'max: {stops["pagerank"].max():.2e}, '
      f'média: {stops["pagerank"].mean():.2e}')

Arestas únicas: 6,974
Nós: 6,426, arestas: 6,974

pagerank - min: 2.80e-05, max: 7.67e-04, média: 1.56e-04


In [7]:
print('Preview das features estáticas:')
stops.head()

Preview das features estáticas:


,stop_id,stop_lat,stop_lon,num_rotas_servindo,pagerank
0,1001O00010C1,-22.897488,-43.186627,2,0.000086
1,1002O00010C0,-22.893310,-43.192600,3,0.000192
2,1002O00013C0,-22.895380,-43.187890,6,0.000219
3,1003O00067C2,-22.894509,-43.197401,5,0.000159
4,1003VZ0052E9,-22.897111,-43.204214,0,0.000028


## 3. Target - tiroteios por (parada, mês) no raio 500m

Para cada tiroteio, achamos todas as paradas dentro de 500m via BallTree haversine. Depois deduplicamos em pares (stop_id, ano_mes).

In [8]:
stops_rad = np.deg2rad(stops[['stop_lat', 'stop_lon']].values)
fogo_rad = np.deg2rad(fogo_rio[['latitude', 'longitude']].values)

tree = BallTree(stops_rad, metric='haversine')
raio_rad = RAIO_TARGET_M / R_TERRA_M

idx_por_tiroteio = tree.query_radius(fogo_rad, r=raio_rad)

# Coletar pares (stop_id, ano_mes) - set garante dedup
pares_positivos = set()
for stop_idxs, ano_mes in zip(idx_por_tiroteio, fogo_rio['ano_mes'].values):
    for si in stop_idxs:
        pares_positivos.add((stops.at[si, 'stop_id'], ano_mes))

print(f'Pares positivos (parada, mês) no raio {RAIO_TARGET_M}m: {len(pares_positivos):,}')

target_df = pd.DataFrame(list(pares_positivos), columns=['stop_id', 'ano_mes'])
target_df['y'] = 1
target_df.head()

Pares positivos (parada, mês) no raio 500m: 71,563


,stop_id,ano_mes,y
0,bu5b,2021-02,1
1,1011O00007C0,2023-04,1
2,3113O00020C0,2023-06,1
3,3063O00082P0,2020-09,1
4,5162O00012C0,2020-07,1


## 4. Janela temporal do dataset

Começa no primeiro mês onde há 12m de histórico completo. Termina no último mês em que as duas bases coexistem.

In [9]:
min_chamados = chamados['ano_mes'].min()
min_fogo = fogo_rio['ano_mes'].min()
max_chamados = chamados['ano_mes'].max()
max_fogo = fogo_rio['ano_mes'].max()

primeiro_mes_dataset = max(min_chamados, min_fogo) + WARMUP_MESES
ultimo_mes_dataset = min(max_chamados, max_fogo)

meses_dataset = pd.period_range(primeiro_mes_dataset, ultimo_mes_dataset, freq='M')

print(f'1746:    {min_chamados} - {max_chamados}')
print(f'Fogo:    {min_fogo} - {max_fogo}')
print(f'Dataset: {primeiro_mes_dataset} - {ultimo_mes_dataset}  ({len(meses_dataset)} meses)')

1746:    2020-01 - 2025-12
Fogo:    2019-12 - 2025-12
Dataset: 2021-01 - 2025-12  (60 meses)


## 5. Features dinâmicas do 1746

Para cada (parada, mês M), contar reclamações vinculadas nos últimos K ∈ {3, 6, 12} meses **antes** de M.

**Estratégia**: para cada categoria de interesse (total, segurança, iluminação), construir matriz `stop_id × ano_mes` com contagem mensal. Rolling sum ao longo do eixo tempo com `shift(1)` garante zero leakage.

In [10]:
# Tipos usados para filtrar cada categoria — copiado de config.TIPO_TO_SERVICO
# para o notebook ser autocontido.
TIPOS_SEGURANCA = [
    'Guarda Municipal / Fiscalização de trânsito',
    'Ordem pública',
    'Ouvidoria SEOP',
    'Patrulhamento público',
]
TIPOS_ILUMINACAO = [
    'Iluminação Pública',
    'Reformulação de iluminação pública',
    'Manutenção de iluminação pública',
]

CATEGORIAS = {
    'total': None,                # sem filtro
    'seguranca': TIPOS_SEGURANCA,
    'iluminacao': TIPOS_ILUMINACAO,
}

def contagens_mensais(df: pd.DataFrame, tipos_filter: list | None) -> pd.DataFrame:
    d = df if tipos_filter is None else df[df['tipo'].isin(tipos_filter)]
    return d.groupby(['stop_id_mais_proximo', 'ano_mes']).size().rename('n').reset_index()

contagens = {cat: contagens_mensais(chamados, filt) for cat, filt in CATEGORIAS.items()}

for cat, df in contagens.items():
    print(f'{cat:12s} -> {len(df):>7,} pares (stop, mês) com >= 1 chamado; total chamados: {df["n"].sum():>7,}')

total        ->  34,830 pares (stop, mês) com >= 1 chamado; total chamados:  47,680
seguranca    ->   2,140 pares (stop, mês) com >= 1 chamado; total chamados:   3,159
iluminacao   ->   5,406 pares (stop, mês) com >= 1 chamado; total chamados:   6,168


In [11]:
# Todos os meses possíveis: do início do warmup até o último mês do dataset.
# Precisamos deste range completo para a rolling ter dados no início.
todos_meses = pd.period_range(
    primeiro_mes_dataset - WARMUP_MESES,
    ultimo_mes_dataset,
    freq='M',
)

JANELAS = [3, 6, 12]
features_1746 = []

for cat, df_cat in contagens.items():
    # Matriz stop × ano_mes com contagem
    pivot = (
        df_cat
        .pivot_table(index='stop_id_mais_proximo', columns='ano_mes',
                     values='n', aggfunc='sum', fill_value=0)
        .reindex(index=stops['stop_id'].values, columns=todos_meses, fill_value=0)
    )
    pivot.index.name = 'stop_id'

    for janela in JANELAS:
        # rolling sum ao longo do tempo (axis=1). shift(1, axis=1) exclui o próprio mês
        # -> a soma para o mês M cobre [M-janela, M-1].
        rolled = pivot.rolling(window=janela, axis=1, min_periods=1).sum().shift(1, axis=1)
        rolled = rolled.loc[:, meses_dataset]  # cortar para o período do dataset
        colname = (
            f'num_reclamacoes_{janela}m' if cat == 'total'
            else f'num_reclamacoes_{cat}_{janela}m'
        )
        long = (
            rolled.reset_index()
                  .melt(id_vars='stop_id', var_name='ano_mes', value_name=colname)
        )
        # Depois do melt, ano_mes vira object; forçar de volta para period[M]
        # para o merge com o grid funcionar (grid tem ano_mes como Period).
        long['ano_mes'] = pd.PeriodIndex(long['ano_mes'], freq='M')
        features_1746.append(long)

print(f'Features 1746 criadas: {len(features_1746)} DataFrames long-format')
print('dtype de ano_mes:', features_1746[0]['ano_mes'].dtype)
features_1746[0].head()

/var/folders/v4/96y4kw3d4kxgn72q44kzbjrc0000gp/T/ipykernel_73455/2586993523.py:25: FutureWarning: Support for axis=1 in DataFrame.rolling is deprecated and will be removed in a future version. Use obj.T.rolling(...) instead
  rolled = pivot.rolling(window=janela, axis=1, min_periods=1).sum().shift(1, axis=1)
/var/folders/v4/96y4kw3d4kxgn72q44kzbjrc0000gp/T/ipykernel_73455/2586993523.py:25: FutureWarning: Support for axis=1 in DataFrame.rolling is deprecated and will be removed in a future version. Use obj.T.rolling(...) instead
  rolled = pivot.rolling(window=janela, axis=1, min_periods=1).sum().shift(1, axis=1)
/var/folders/v4/96y4kw3d4kxgn72q44kzbjrc0000gp/T/ipykernel_73455/2586993523.py:25: FutureWarning: Support for axis=1 in DataFrame.rolling is deprecated and will be removed in a future version. Use obj.T.rolling(...) instead
  rolled = pivot.rolling(window=janela, axis=1, min_periods=1).sum().shift(1, axis=1)
/var/folders/v4/96y4kw3d4kxgn72q44kzbjrc0000gp/T/ipykernel_73455/25869

Features 1746 criadas: 9 DataFrames long-format
dtype de ano_mes: period[M]


/var/folders/v4/96y4kw3d4kxgn72q44kzbjrc0000gp/T/ipykernel_73455/2586993523.py:25: FutureWarning: Support for axis=1 in DataFrame.rolling is deprecated and will be removed in a future version. Use obj.T.rolling(...) instead
  rolled = pivot.rolling(window=janela, axis=1, min_periods=1).sum().shift(1, axis=1)


,stop_id,ano_mes,num_reclamacoes_3m
0,1001O00010C1,2021-01,0.0
1,1002O00010C0,2021-01,0.0
2,1002O00013C0,2021-01,0.0
3,1003O00067C2,2021-01,0.0
4,1003VZ0052E9,2021-01,0.0


## 6. Montagem do dataset

In [12]:
# Grid completo: uma linha por (parada, mês)
grid = pd.MultiIndex.from_product(
    [stops['stop_id'].values, meses_dataset],
    names=['stop_id', 'ano_mes'],
).to_frame(index=False)

print(f'Grid base: {len(grid):,} linhas ({len(stops):,} paradas × {len(meses_dataset)} meses)')

Grid base: 385,560 linhas (6,426 paradas × 60 meses)


In [13]:
# Merge features estáticas
dataset = grid.merge(
    stops[['stop_id', 'stop_lat', 'stop_lon', 'num_rotas_servindo', 'pagerank']].rename(
        columns={'stop_lat': 'lat', 'stop_lon': 'lon'}
    ),
    on='stop_id',
    how='left',
)
print(f'Após estáticas: {dataset.shape}')

Após estáticas: (385560, 6)


In [14]:
# Merge features do 1746 (9 features: total 3/6/12, seg 3/6/12, ilum 3/6/12)
for df_feat in features_1746:
    dataset = dataset.merge(df_feat, on=['stop_id', 'ano_mes'], how='left')

# Contagens vindas do rolling podem ser float por causa de shift; forçar int
for col in dataset.columns:
    if col.startswith('num_reclamacoes_'):
        dataset[col] = dataset[col].fillna(0).astype(int)

print(f'Após 1746: {dataset.shape}')

Após 1746: (385560, 15)


In [15]:
# Features temporais brutas (dependem só de ano_mes).
# A codificação cíclica (mes_sin/mes_cos) fica no notebook do modelo — é decisão
# de modelagem, não estruturação do dataset bruto.
dataset['mes'] = dataset['ano_mes'].dt.month
dataset['ano'] = dataset['ano_mes'].dt.year

print(f'Após temporais: {dataset.shape}')

Após temporais: (385560, 17)


In [16]:
# Target
dataset = dataset.merge(target_df, on=['stop_id', 'ano_mes'], how='left')
dataset['y'] = dataset['y'].fillna(0).astype(int)

print(f'Final: {dataset.shape}')
print()
print('Distribuição do target:')
print(dataset['y'].value_counts())
print(f'\nProporção de positivos: {100*dataset["y"].mean():.2f}%')

Final: (385560, 18)

Distribuição do target:
y
0    327889
1     57671
Name: count, dtype: int64

Proporção de positivos: 14.96%


## 7. Sanity checks

In [17]:
null_counts = dataset.isnull().sum()
nulls_nonzero = null_counts[null_counts > 0]
print('Nulos por coluna:')
if len(nulls_nonzero):
    print(nulls_nonzero)
else:
    print('  Nenhum.')

Nulos por coluna:
  Nenhum.


In [18]:
print('Tipos e memória:')
dataset.info()

Tipos e memória:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 385560 entries, 0 to 385559
Data columns (total 18 columns):
 #   Column                          Non-Null Count   Dtype    
---  ------                          --------------   -----    
 0   stop_id                         385560 non-null  object   
 1   ano_mes                         385560 non-null  period[M]
 2   lat                             385560 non-null  float64  
 3   lon                             385560 non-null  float64  
 4   num_rotas_servindo              385560 non-null  int64    
 5   pagerank                        385560 non-null  float64  
 6   num_reclamacoes_3m              385560 non-null  int64    
 7   num_reclamacoes_6m              385560 non-null  int64    
 8   num_reclamacoes_12m             385560 non-null  int64    
 9   num_reclamacoes_seguranca_3m    385560 non-null  int64    
 10  num_reclamacoes_seguranca_6m    385560 non-null  int64    
 11  num_reclamacoes_seguranca_12m   385

In [19]:
feature_cols = [c for c in dataset.columns if c not in ('stop_id', 'ano_mes', 'y')]
dataset[feature_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
lat,385560.0,-22.900446,0.055326,-23.033801,-22.932856,-22.893884,-22.860910,-22.784874
lon,385560.0,-43.333426,0.102080,-43.563220,-43.396232,-43.327521,-43.253930,-43.160810
num_rotas_servindo,385560.0,3.895580,4.206934,0.000000,1.000000,3.000000,5.000000,60.000000
pagerank,385560.0,0.000156,0.000089,0.000028,0.000103,0.000151,0.000201,0.000767
num_reclamacoes_3m,385560.0,0.298472,1.118802,0.000000,0.000000,0.000000,0.000000,73.000000
num_reclamacoes_6m,385560.0,0.597297,1.925455,0.000000,0.000000,0.000000,1.000000,140.000000
num_reclamacoes_12m,385560.0,1.186259,3.333531,0.000000,0.000000,0.000000,1.000000,157.000000
num_reclamacoes_seguranca_3m,385560.0,0.020134,0.363771,0.000000,0.000000,0.000000,0.000000,43.000000
num_reclamacoes_seguranca_6m,385560.0,0.041454,0.665590,0.000000,0.000000,0.000000,0.000000,69.000000
num_reclamacoes_seguranca_12m,385560.0,0.086360,1.238832,0.000000,0.000000,0.000000,0.000000,134.000000


In [20]:
# Distribuição temporal do target — sanity check: proporção deve ser relativamente
# estável ao longo dos meses (sem saltos bruscos que indicariam bug)
por_mes = dataset.groupby('ano_mes').agg(
    total_paradas=('stop_id', 'count'),
    positivos=('y', 'sum'),
)
por_mes['pct_positivos'] = 100 * por_mes['positivos'] / por_mes['total_paradas']
print('Por mês (primeiros e últimos):')
print(por_mes.head(6))
print('...')
print(por_mes.tail(6))

Por mês (primeiros e últimos):
         total_paradas  positivos  pct_positivos
ano_mes                                         
2021-01           6426       1065      16.573296
2021-02           6426       1162      18.082789
2021-03           6426       1517      23.607221
2021-04           6426       1190      18.518519
2021-05           6426       1349      20.992842
2021-06           6426       1177      18.316215
...
         total_paradas  positivos  pct_positivos
ano_mes                                         
2025-07           6426        664      10.333022
2025-08           6426        823      12.807345
2025-09           6426        738      11.484594
2025-10           6426        674      10.488640
2025-11           6426        671      10.441955
2025-12           6426        726      11.297852


In [21]:
# Preview de linhas positivas para conferir se as features fazem sentido
dataset[dataset['y'] == 1].head(5)

,stop_id,ano_mes,lat,lon,num_rotas_servindo,pagerank,num_reclamacoes_3m,num_reclamacoes_6m,num_reclamacoes_12m,num_reclamacoes_seguranca_3m,num_reclamacoes_seguranca_6m,num_reclamacoes_seguranca_12m,num_reclamacoes_iluminacao_3m,num_reclamacoes_iluminacao_6m,num_reclamacoes_iluminacao_12m,mes,ano,y
7,1001O00010C1,2021-08,-22.897488,-43.186627,2,0.000086,1,1,1,0,0,0,0,0,0,8,2021,1
26,1001O00010C1,2023-03,-22.897488,-43.186627,2,0.000086,0,0,1,0,0,0,0,0,0,3,2023,1
51,1001O00010C1,2025-04,-22.897488,-43.186627,2,0.000086,0,2,2,0,0,0,0,0,0,4,2025,1
62,1002O00010C0,2021-03,-22.893310,-43.192600,3,0.000192,0,0,0,0,0,0,0,0,0,3,2021,1
75,1002O00010C0,2022-04,-22.893310,-43.192600,3,0.000192,5,6,6,0,0,0,0,0,0,4,2022,1


## 8. Persistência

In [22]:
# ano_mes é Period, precisa virar string para o Parquet aceitar
dataset_export = dataset.copy()
dataset_export['ano_mes'] = dataset_export['ano_mes'].astype(str)

OUTPUT.parent.mkdir(parents=True, exist_ok=True)
dataset_export.to_parquet(OUTPUT, index=False, compression='snappy')

print(f'Salvo em: {OUTPUT}')
print(f'Tamanho: {OUTPUT.stat().st_size / 1024 / 1024:.2f} MB')
print(f'Linhas:  {len(dataset_export):,}')
print(f'Colunas ({len(dataset_export.columns)}): {list(dataset_export.columns)}')

Salvo em: /Users/vinicius.maciel/CascadeProjects/poli/riomobianalytics/data/model_dataset.parquet
Tamanho: 0.66 MB
Linhas:  385,560
Colunas (18): ['stop_id', 'ano_mes', 'lat', 'lon', 'num_rotas_servindo', 'pagerank', 'num_reclamacoes_3m', 'num_reclamacoes_6m', 'num_reclamacoes_12m', 'num_reclamacoes_seguranca_3m', 'num_reclamacoes_seguranca_6m', 'num_reclamacoes_seguranca_12m', 'num_reclamacoes_iluminacao_3m', 'num_reclamacoes_iluminacao_6m', 'num_reclamacoes_iluminacao_12m', 'mes', 'ano', 'y']
